In [ ]:
from pathlib import Path
import sys
import pandas as pd
import os
import imageio_ffmpeg

ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
ffmpeg_dir = str(Path(ffmpeg_exe).parent)

# Make sure subprocess calls can find it
os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ.get("PATH", "")

# Tell MoviePy explicitly (uses imageio-ffmpeg binary)
from moviepy.config import change_settings
change_settings({"FFMPEG_BINARY": ffmpeg_exe})

print("FFmpeg ->", ffmpeg_exe)

from moviepy.editor import VideoFileClip

# Find project root (the folder that contains "src")
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = cwd.parent  # if you're inside notebooks/, this is project/

SRC_PATH = (PROJECT_ROOT / "scripts/data_preparation").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
    print(f"Added to sys.path: {SRC_PATH}")

ORIG_VIDEO_PATH = (PROJECT_ROOT / "dataset/original_videos").resolve()
VIDEO_CLIP_PATH = (PROJECT_ROOT / "dataset/clips_video").resolve()
AUDIO_CLIP_PATH = (PROJECT_ROOT / "dataset/clips_audio").resolve()
CSV_PATH        = (PROJECT_ROOT / "dataset/csvs").resolve()
OUT_PATH        = (PROJECT_ROOT / "dataset/output").resolve()

from extract_video_data import (
    extract_frames_grid,
    pick_uniform_times,
    transcribe_whisper,
    write_srt,
    extract_audio
)

In [ ]:
yt_df = pd.read_csv((CSV_PATH / "dataset.csv").resolve())
vid_ids = set(yt_df["Video_ID"].astype(str))
file_ids = set(f.stem for f in VIDEO_CLIP_PATH.glob("*.mp4")) 
present_ids = vid_ids.intersection(file_ids)
missing_ids = vid_ids - file_ids

{behaviors}
Absence or Avoidance of Eye Contact
Aggressive Behavior
Hyper- or Hyporeactivity to Sensory Input
Non-Responsiveness to Verbal Interaction
Non-Typical Language
Object Lining-Up
Self-Hitting or Self-Injurious Behavior
Self-Spinning or Spinning Objects
Upper Limb Stereotypies
Background (i.e., not-applicable)

In [ ]:
frames = 100
grid_rows, grid_cols = 10, 10
assert frames == grid_rows * grid_cols

success_ids = []
failure_ids = []
for vid_id in present_ids:
    # if vid_id!='x-u1JmAxWUI_929_945': continue
    vid_path = (VIDEO_CLIP_PATH / f"{vid_id}.mp4").resolve()
    aud_path = (VIDEO_CLIP_PATH / f"{vid_id}.m4a").resolve()
    if not vid_path.exists():
        print(f"⚠️ Video not found: {vid_path}")
        failure_ids.append(vid_id)
        continue

    # I am not using the audio clip, rather extracting the audio form the video file
    # aud_path = (AUDIO_CLIP_PATH / f"{vid_id}.wav").resolve()
    # if not aud_path.exists():
    #     print(f"⚠️ Video not found: {aud_path}")
    #     failure_ids.append(vid_id)
    #     continue
    
    clip=None
    try:
        # ---- Video ----
        clip = VideoFileClip(str(vid_path))
        duration = float(clip.duration or 0.0)
        times = pick_uniform_times(duration, frames)

        grid_img = extract_frames_grid(
            clip=clip,
            times=times,
            grid_rows=grid_rows,
            grid_cols=grid_cols,
            cell_width=None,
            cell_height=None,
        )

        id_out_dir = (OUT_PATH / str(vid_id)).resolve()
        id_out_dir.mkdir(parents=True, exist_ok=True)

        grid_path = id_out_dir / f"{vid_id}_grid_{grid_rows}x{grid_cols}.jpg"
        grid_img.save(grid_path, quality=95, subsampling=1)
        print(f"🆗 Saved grid for {vid_id} -> {grid_path}")

        # ---- Audio Extraction ----        
        extract_audio(clip, aud_path, bitrate="192k")

        # ---- Audio / Transcription ----
        transcript_text, segments = transcribe_whisper(
            audio_path=aud_path,
            model_name="base",
            device=None,
            language=None,
        )

        # Save TXT
        txt_path = id_out_dir / f"{vid_id}_transcript.txt"
        txt_path.write_text((transcript_text or "").strip() + "\n", encoding="utf-8")

        # Save SRT
        srt_path = id_out_dir / f"{vid_id}_srt.srt"
        if segments:
            write_srt(segments, srt_path)

        print(f"🆗 Saved transcript and SRT for {vid_id}")
        success_ids.append(vid_id)

    except Exception as e:
        print(f"❌ Failed processing {vid_id}: {e}")

    finally:
        if clip is not None:
            clip.close()

print("\n=== SUMMARY ===")
print(f"✅ Success: {len(success_ids)} ids")
print(f"❌ Failure: {len(failure_ids)} ids")

#### Straight dump images to CHRI

In [5]:
from pathlib import Path

# Output directory (flat)
OUT_DIR = Path("/home/anirban/projects/AV-ASD/chri/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

frames = 25
grid_rows, grid_cols = 5, 5
assert frames == grid_rows * grid_cols

success_ids = []
failure_ids = []

for vid_id in present_ids:
    vid_path = (VIDEO_CLIP_PATH / f"{vid_id}.mp4").resolve()

    if not vid_path.exists():
        print(f"⚠️ Video not found: {vid_path}")
        failure_ids.append(vid_id)
        continue

    clip = None
    try:
        clip = VideoFileClip(str(vid_path))
        duration = float(clip.duration or 0.0)
        times = pick_uniform_times(duration, frames)

        grid_img = extract_frames_grid(
            clip=clip,
            times=times,
            grid_rows=grid_rows,
            grid_cols=grid_cols,
            cell_width=None,
            cell_height=None,
        )

        # Save directly into /chri with filename only
        out_file = OUT_DIR / f"{vid_id}_grid_{grid_rows}x{grid_cols}.jpg"
        grid_img.save(out_file, quality=95, subsampling=1)

        print(f"✅ Saved: {out_file}")
        success_ids.append(vid_id)

    except Exception as e:
        print(f"❌ Failed {vid_id}: {e}")
        failure_ids.append(vid_id)
    
    finally:
        if clip is not None:
            clip.close()

print("\n=== DONE ===")
print(f"✅ Success: {len(success_ids)}")
print(f"❌ Failures: {len(failure_ids)}")


✅ Saved: /home/anirban/projects/AV-ASD/chri/VWmNcRO9tgk_88_91_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/VWmNcRO9tgk_170_173_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/YtvP5A5OHpU_232_243_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/Vwlc3fLmipY_0_4_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/1fa9Q8sTD4w_0_57_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/OETvkJpwVlY_325_335_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/VWmNcRO9tgk_211_214_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/441244708182765_153_155_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/8N2aiSrfGeE_306_311_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/5sgfS0SSh8o_32_46_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/VWmNcRO9tgk_101_104_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/bDQs4Ckftxs_100_110_grid_5x5.jpg
✅ Saved: /home/anirban/projects/AV-ASD/chri/VWmNcRO9tgk_77_82_grid_5x5.jpg
✅ Saved: